# Free AI Server on Kaggle — Ollama + Muse-Glimmer-30B + Tailscale

Ye notebook ek free **Kaggle GPU session** ko private AI server bana deta hai — 2× Tesla T4 GPUs pe **Muse-Glimmer-30B** (Q4_K_M) chalake.

Access **Tailscale** se milega — Cloudflare tunnel NAHI. Faida: URL private hai (sirf tere devices), aur **IP har session me SAME rehta hai** (Cloudflare har baar naya URL deta tha).

### Kya-milega
- Ollama + `hf.co/meta-models/Muse-Glimmer-30B-GGUF:Q4_K_M` (~18-19 GB)
- Model dono T4 GPUs me split
- 64K context + MTP (speculative decoding)
- Tailscale private URL — `http://100.x.y.z:11434`

### Shuru karne se pehle (Kaggle me)
1. Right side panel: **Settings → Accelerator → GPU T4 ×2**
2. **Settings → Internet → ON** (zaroori!)
3. **Run All** daba do (ya upar se ek-ek cell chalao)

> **Dhyan do:** Pehla download ~18-19 GB hai — 15-25 minute. Kaggle session band hote hi sab gaya; naya session me ye notebook dobara Run karo (IP same milega).

## 1. Check the GPU

Ye chalao — **do Tesla T4 GPUs** dikhne chahiye.

In [ ]:
!nvidia-smi

## 2. Install Ollama

Har naye Kaggle session me ek baar chalao.

In [ ]:
# Install Ollama
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama
!ollama --version


## 3. Start Ollama (64K context config)

Model `/kaggle/working/ollama-models` me store hoga aur inference ke baad loaded rahega. Server `127.0.0.1:11434` pe sunega — Tailscale usi ko private tunnel karega.

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

MODEL_REPO = "hf.co/meta-models/Muse-Glimmer-30B-GGUF:Q4_K_M"
MTP_MODEL = "muse-glimmer-30b-mtp"

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODELS = Path("/kaggle/working/ollama-models")

NUM_CTX = 65536
DRAFT_TOKENS = 2

OLLAMA_MODELS.mkdir(parents=True, exist_ok=True)
os.environ["OLLAMA_MODELS"] = str(OLLAMA_MODELS)

# Ollama missing ho to khud install (fresh session resilience)
OLLAMA_BIN = shutil.which("ollama")
if not OLLAMA_BIN:
    print("Ollama executable not found. Installing it now...")
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True,
    )
    OLLAMA_BIN = shutil.which("ollama")

if not OLLAMA_BIN:
    raise RuntimeError(
        "Ollama installation completed but the executable could not be found."
    )

OLLAMA_ENV = os.environ.copy()
OLLAMA_ENV.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": str(OLLAMA_MODELS),
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_CONTEXT_LENGTH": str(NUM_CTX),
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
})

def ollama_ready():
    try:
        urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2).close()
        return True
    except Exception:
        return False

if not ollama_ready():
    server_log = open("/tmp/ollama-server.log", "a")
    ollama_server_process = subprocess.Popen(
        [OLLAMA_BIN, "serve"],
        env=OLLAMA_ENV,
        stdout=server_log,
        stderr=subprocess.STDOUT,
    )

    for _ in range(60):
        if ollama_ready():
            break
        time.sleep(1)
    else:
        server_log.close()
        raise RuntimeError(
            "Ollama did not become ready. Check /tmp/ollama-server.log"
        )

print("✅ Ollama is ready:", OLLAMA_URL)
print("✅ Model:", MODEL_REPO)
print("✅ Model directory:", OLLAMA_MODELS)
print("✅ Context length:", NUM_CTX)


## 4. Download Muse-Glimmer-30B (≈18-19 GB, 15-25 min)

Model seedha Ollama ke store me download hota hai. Progress % dikhta rahega — **beech me interrupt mat karna**.

Pehla pull normal hota hai; `--insecure` fallback un Ollama builds ke liye hai jo Hugging Face/Xet ke redirect error `blocked redirect to a different host` dete hain.

In [ ]:
import subprocess

result = subprocess.run([OLLAMA_BIN, "pull", MODEL_REPO], env=OLLAMA_ENV)
if result.returncode != 0:
    print("Standard pull failed; retrying with --insecure...")
    subprocess.run(
        [OLLAMA_BIN, "pull", "--insecure", MODEL_REPO],
        env=OLLAMA_ENV,
        check=True,
    )

print("")
print("✅ Model is available:")
subprocess.run([OLLAMA_BIN, "list"], env=OLLAMA_ENV, check=True)


## 5. Confirm the model is installed

In [ ]:
!{OLLAMA_BIN} list

## 6. Load the base model and verify GPU execution

Ollama ko `100% GPU` report karna chahiye aur dono T4s me model memory dikhni chahiye.

In [ ]:
!{OLLAMA_BIN} run "hf.co/meta-models/Muse-Glimmer-30B-GGUF:Q4_K_M" "Reply with exactly: Model is working."

In [ ]:
!{OLLAMA_BIN} ps

In [ ]:
!nvidia-smi

## 7. Create the MTP model

Already-downloaded base model ko reuse karke MTP parameters wala model banate hain.

In [ ]:
import json
import urllib.request

payload = {
    "model": MTP_MODEL,
    "from": MODEL_REPO,
    "parameters": {
        "draft_num_predict": DRAFT_TOKENS,
        "num_ctx": NUM_CTX,
    },
    "stream": False,
}
request = urllib.request.Request(
    f"{OLLAMA_URL}/api/create",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=300) as response:
    result = json.loads(response.read())
print(result)

## 8. Load and pin the MTP model

In [ ]:
import json
import urllib.request

payload = {
    "model": MTP_MODEL,
    "prompt": "Reply with exactly: MTP is working.",
    "stream": False,
    "keep_alive": -1,
    "options": {
        "draft_num_predict": DRAFT_TOKENS,
        "num_ctx": NUM_CTX,
    },
}
request = urllib.request.Request(
    f"{OLLAMA_URL}/api/generate",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=600) as response:
    result = json.loads(response.read())
print("Response:", result.get("response", "").strip())
print("MTP model loaded and pinned.")

In [ ]:
!{OLLAMA_BIN} ps

In [ ]:
!nvidia-smi

## 9. Benchmark generation speed

Simple session-level measurement; prompt aur hardware conditions ke hisaab se badal sakta hai.

In [ ]:
import json
import time
import urllib.request

prompt = "Explain in about 200 words why GPUs are useful for running large language models. Cover memory bandwidth, parallel computation, and matrix multiplication."
payload = {
    "model": MTP_MODEL,
    "prompt": prompt,
    "think": False,
    "stream": False,
    "keep_alive": -1,
    "options": {
        "draft_num_predict": DRAFT_TOKENS,
        "num_ctx": NUM_CTX,
        "temperature": 0,
    },
}
start = time.perf_counter()
request = urllib.request.Request(
    f"{OLLAMA_URL}/api/generate",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=600) as response:
    result = json.loads(response.read())
wall_time = time.perf_counter() - start
eval_count = result.get("eval_count", 0)
eval_seconds = result.get("eval_duration", 0) / 1e9
speed = eval_count / eval_seconds if eval_seconds else 0
print(result.get("response", "").strip())
print("")
print("--- Benchmark ---")
print(f"Generated tokens: {eval_count}")
print(f"Generation time: {eval_seconds:.2f}s")
print(f"Wall time: {wall_time:.2f}s")
print(f"Generation speed: {speed:.2f} tokens/s")

# 🔒 Private access with Tailscale (Cloudflare ki jagah)

**Tailscale = ek private network jisme sirf TERE devices hain.** Cloudflare quick tunnel jaisa public URL nahi milta — matlab koi ajnabi URL pakad ke tera server use NAHI kar sakta. Aur sabse bada faida: **Tailscale IP har session me same rehta hai (100.x.y.z)** — Cloudflare wali tarah har baar naya URL copy nahi karna padta.

### Kaise chalega (3 chhote cells)
1. **Cell 10** — Tailscale install (1-2 min)
2. **Cell 11** — ek **LOGIN LINK** print hoga → us pe click karke Google/GitHub se login (pehli baar me account banana pad sakta hai — free hai, card nahi maangta)
3. **Cell 12** — tera permanent IP + URL print

> **Note:** PC pe Tailscale already installed hai (Orthos cloud memory usi se chalti hai) — PC side kuch naya install nahi karna.

## 10. Install Tailscale

In [ ]:
import subprocess

def sh(cmd, timeout=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)

r = sh("which tailscale")
if r.returncode != 0:
    print("Tailscale install ho raha hai... (1-2 minute)")
    r = sh("curl -fsSL https://tailscale.com/install.sh | sh")
    if r.returncode != 0:
        print(r.stdout[-2000:])
        print(r.stderr[-2000:])
        raise RuntimeError("Tailscale install fail — Kaggle me Internet ON hai na? (Settings → Internet)")

v = sh("tailscale version")
first = (v.stdout or v.stderr).splitlines()[0] if (v.stdout or v.stderr) else "?"
print("✅ Tailscale installed:", first)


## 11. Start Tailscale + Login

Ye cell chalao — ek **login link** print hoga. Link pe click karo → Google login → bas. Cell khud wait karega jab tak login complete na ho.

In [ ]:
import os
import subprocess
import time

def sh(cmd, timeout=60):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)

st = sh("tailscale status")
if st.returncode == 0 and "100." in st.stdout:
    print("✅ Pehle se logged in hai — seedha agla cell chalao.")
else:
    if st.returncode != 0:
        # Daemon chalu nahi — userspace-networking mode me start karo
        # (Kaggle containers me TUN device nahi hota, ye mode wahan bhi chalta hai)
        state_dir = "/kaggle/working/tailscale_state"
        os.makedirs(state_dir, exist_ok=True)
        daemon_log = open("/tmp/tailscaled.log", "a")
        subprocess.Popen(
            [
                "tailscaled",
                "--tun=userspace-networking",
                "--state", os.path.join(state_dir, "tailscaled.state"),
                "--statedir", state_dir,
                "--socket", "/var/run/tailscale/tailscaled.sock",
            ],
            stdout=daemon_log,
            stderr=subprocess.STDOUT,
        )
        time.sleep(4)

    print("Login ka wait kar raha hoon... Link aate hi us pe click karo 👇")
    print("")
    up = subprocess.Popen(
        ["tailscale", "up", "--accept-dns=false"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in up.stdout:
        line = line.strip()
        if not line:
            continue
        if "login.tailscale.com" in line:
            print("=" * 62)
            print("👆 YAHAN CLICK KARO:", line)
            print("   (Google / GitHub / Microsoft — kuch se bhi login ho jaao)")
            print("=" * 62)
        else:
            print(line)
    up.wait(timeout=900)
    print("")
    print("✅ Login done! Ab agla cell chalao.")


## 12. Server ready — URL yahan milega ⭐

In [ ]:
import subprocess
import time

def sh(cmd, timeout=30):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)

ip = ""
for _ in range(45):
    r = sh("tailscale ip -4")
    if r.returncode == 0:
        lines = [l.strip() for l in r.stdout.splitlines() if l.strip().startswith("100.")]
        if lines:
            ip = lines[0]
            break
    time.sleep(2)

if not ip:
    raise RuntimeError("IP nahi mila — upar wala login cell dobara chalao aur link pe click karo.")

TAILSCALE_URL = f"http://{ip}:11434"

print("=" * 62)
print("🎉 SERVER READY! Ye 3 cheezein note kar lo:")
print("=" * 62)
print("")
print("  Orthos / Ollama URL : " + TAILSCALE_URL)
print("  OpenAI Base URL     : " + TAILSCALE_URL + "/v1")
print("  Model name          : " + MTP_MODEL)
print("")
print("-" * 62)
print("PC se test (cmd me): curl " + TAILSCALE_URL + "/api/tags")
print("-" * 62)
print("")
print(sh("tailscale status").stdout)


## 13. Test the API (local)

In [ ]:
import json
import urllib.request

def chat(prompt, base_url=OLLAMA_URL, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    payload = {
        "model": MTP_MODEL,
        "messages": messages,
        "stream": False,
        "keep_alive": -1,
        "options": {
            "draft_num_predict": DRAFT_TOKENS,
            "num_ctx": NUM_CTX,
        },
    }
    request = urllib.request.Request(
        f"{base_url.rstrip('/')}/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=600) as response:
        result = json.loads(response.read())
    return result["message"]["content"]

print("Local API test:", chat("Write a one-line Python hello-world program."))
print("")
print("(Asli Tailscale test PC se hota hai — cmd me: curl " + TAILSCALE_URL + "/api/tags")

## 🖥️ Orthos (PC) ko jodo — sirf ek baar

1. PC pe Tailscale **already installed** hai (Orthos cloud memory usi pe chalti hai) — kuch install nahi karna
2. Orthos me LLM provider switch karo aur URL daalo (Cell 12 me jo print hua):
   - **URL:** `http://100.x.y.z:11434`
   - **Model:** `muse-glimmer-30b-mtp`
3. PC se quick test (cmd me): `curl http://100.x.y.z:11434/api/tags`

### OpenAI-compatible tools ke liye
- **Base URL:** `http://100.x.y.z:11434/v1`
- **API key:** kuch bhi (jaise `ollama`)
- **Model:** `muse-glimmer-30b-mtp`

> ⚠️ Kaggle session band = server band. Naya session = ye notebook dobara Run All (model phir download hoga; **Tailscale IP same rahega**, state `/kaggle/working/tailscale_state` me save hota hai).

## Interactive Ollama CLI (optional)

Sab setup complete hone ke baad hi chalao. Exit karne ke liye `/bye`.

In [ ]:
!{OLLAMA_BIN} run muse-glimmer-30b-mtp